# Indicator variables

A category is entered into a linear model by indicators, one fewer than the number of categories when an intercept is present. The coefficients are then differences of group means. Including one indicator per category together with an intercept makes $X$ singular.


## Two groups

Four responses and a group label:

$$
\begin{array}{c|cccc}
\text{group} & A & A & B & B \\
\hline
y & 2 & 4 & 5 & 7
\end{array}
$$

Let $d = 0$ for group $A$ and $d = 1$ for group $B$, and fit $y = b_0 + b_1 d$. The group means are

$$
\bar y_A = 3, \qquad \bar y_B = 6.
$$

The least-squares fit of a binary predictor reproduces those means:

$$
b_0 = 3, \qquad b_1 = 6 - 3 = 3, \qquad \widehat y = 3 + 3d.
$$

Predictions are $3, 3, 6, 6$. Residuals are $-1, 1, -1, 1$, and

$$
\mathrm{RSS} = 4.
$$

The slope is not a change "per unit of a continuous $x$." It is the difference between the two group means.


In [1]:
# A single indicator recovers the two group means.
import numpy as np
group = ["A", "A", "B", "B"]
y = np.array([2.0, 4, 5, 7])
d = np.array([0.0, 0, 1, 1])
X = np.column_stack([np.ones(4), d])
beta = np.linalg.solve(X.T @ X, X.T @ y)
residuals = y - X @ beta
print("beta", beta)
print("RSS", np.sum(residuals ** 2))
assert np.allclose(beta, [3, 3])
assert np.sum(residuals ** 2) == 4
assert abs(y[:2].mean() - 3) < 1e-12
assert abs(y[2:].mean() - 6) < 1e-12


beta [3. 3.]
RSS 4.0


## Three groups, two indicators

Add group $C$ with responses $1$ and $3$, and keep $A$ as the reference category. The means are $\bar y_C = 2$, $\bar y_A = 3$, and $\bar y_B = 6$. Fit

$$
y = b_0 + b_B d_B + b_C d_C.
$$

Then $b_0$ is the reference mean, and the other coefficients are differences from it:

$$
b_0 = 3, \qquad b_B = 3, \qquad b_C = 2 - 3 = -1.
$$

A member of $C$ is predicted by $3 + (-1) = 2$. A negative coefficient is a lower mean than the reference group, not a failed fit.


In [2]:
# Two indicators, with group A as the reference category.
import numpy as np
y = np.array([2.0, 4, 5, 7, 1, 3])
d_b = np.array([0, 0, 1, 1, 0, 0], dtype=float)
d_c = np.array([0, 0, 0, 0, 1, 1], dtype=float)
X = np.column_stack([np.ones(6), d_b, d_c])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("beta", beta)
print("predictions", X @ beta)
assert np.allclose(beta, [3, 3, -1])
assert np.allclose(X @ beta, [3, 3, 6, 6, 2, 2])


beta [ 3.  3. -1.]
predictions [3. 3. 6. 6. 2. 2.]


## One change: a different reference group

Code group $B$ as the reference instead. The new coefficients are $b_0 = 6$, $b_A = -3$, and $b_C = -4$. The predictions are unchanged: group $A$ is still predicted by $6 + (-3) = 3$, and group $C$ by $6 + (-4) = 2$. Changing the reference renames the same means as different differences. It does not change the fitted values.


In [3]:
# A new reference group rewrites the coefficients and keeps the predictions.
import numpy as np
y = np.array([2.0, 4, 5, 7, 1, 3])
d_a = np.array([1, 1, 0, 0, 0, 0], dtype=float)
d_c = np.array([0, 0, 0, 0, 1, 1], dtype=float)
X = np.column_stack([np.ones(6), d_a, d_c])
beta = np.linalg.solve(X.T @ X, X.T @ y)
print("beta with B as reference", beta)
print("predictions", X @ beta)
assert np.allclose(beta, [6, -3, -4])
assert np.allclose(X @ beta, [3, 3, 6, 6, 2, 2])


beta with B as reference [ 6. -3. -4.]
predictions [3. 3. 6. 6. 2. 2.]


## Pitfall: the dummy trap

For the two-group table, form both indicators and also keep the intercept. Every row then satisfies $d_A + d_B = 1$, so the three columns of

$$
X = \begin{bmatrix}
1 & 1 & 0 \\
1 & 1 & 0 \\
1 & 0 & 1 \\
1 & 0 & 1
\end{bmatrix}
$$

are linearly dependent. The Gram matrix is singular and the normal equations have infinitely many solutions: adding $c$ to the intercept and subtracting $c$ from both group coefficients leaves $X\beta$ unchanged. Software that tries to invert $X^{\mathsf T}X$ should fail, not pick one of those solutions and call it the answer.


In [4]:
# An intercept plus one indicator per group makes X'X singular.
import numpy as np
X = np.array([
    [1.0, 1, 0],
    [1, 1, 0],
    [1, 0, 1],
    [1, 0, 1],
])
gram = X.T @ X
print("X'X")
print(gram)
print("determinant", np.linalg.det(gram))
assert abs(np.linalg.det(gram)) < 1e-10
try:
    np.linalg.solve(gram, np.array([18.0, 6, 12]))
    raise AssertionError("a singular system should not solve")
except np.linalg.LinAlgError:
    print("normal equations refused: LinAlgError")


X'X
[[4. 2. 2.]
 [2. 2. 0.]
 [2. 0. 2.]]
determinant 0.0
normal equations refused: LinAlgError


## Summary

- With an intercept, use one fewer indicator than there are categories.
- The intercept is the reference mean. Each coefficient is a difference of means.
- An indicator for every category plus an intercept makes the columns sum to a dependence, and $X^{\mathsf T}X$ is singular.
